In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

TARGET = "diagnosis"
RANDOM_STATE = 42
TEST_SIZE = 0.30
MAX_DEPTH = 4

df = pd.read_csv("breast-cancer-mean-minmax.csv")

df[TARGET] = df[TARGET].map({"B": 0, "M": 1})

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

In [2]:
correlation = df.corr(numeric_only=True)

target_corr = correlation[TARGET].drop(TARGET)

selected_features = target_corr[abs(target_corr) > 0.6].index.tolist()

print("Correlation with diagnosis")
print(target_corr.sort_values(ascending=False))

print("\nSelected Features")
print(selected_features)

print(f"\nNumber of selected features: {len(selected_features)}")

Correlation with diagnosis
concave points_worst       0.790797
perimeter_worst            0.782914
radius_worst               0.776454
concave points_mean        0.768315
perimeter_mean             0.742636
area_worst                 0.733825
radius_mean                0.730029
area_mean                  0.708984
concavity_mean             0.686318
concavity_worst            0.649154
compactness_mean           0.596534
compactness_worst          0.590998
radius_se                  0.567134
perimeter_se               0.556141
area_se                    0.548236
texture_worst              0.456903
smoothness_worst           0.421465
symmetry_worst             0.416294
texture_mean               0.415185
concave points_se          0.390662
smoothness_mean            0.358560
symmetry_mean              0.330499
fractal_dimension_worst    0.323872
compactness_se             0.292999
concavity_se               0.237806
fractal_dimension_se       0.077972
id                         0.039769
s

In [3]:
N_COMPONENTS = len(selected_features)

pca = PCA(n_components=N_COMPONENTS)

X_train_pca = pca.fit_transform(X_train)

X_test_pca = pca.transform(X_test)
print("\nExplained Variance Ratio")
print(pca.explained_variance_ratio_)

print("\nTotal Explained Variance")
print(f"{sum(pca.explained_variance_ratio_)*100:.2f}%")


Explained Variance Ratio
[0.49811606 0.17397863 0.06913948 0.06607031 0.03951197 0.03406827
 0.02944478 0.01694182 0.01149777 0.01027356]

Total Explained Variance
94.90%


In [4]:
model = DecisionTreeClassifier(
    max_depth=MAX_DEPTH,
    random_state=RANDOM_STATE
)

model.fit(X_train, y_train)

originalAcc = accuracy_score(
    y_test,
    model.predict(X_test)
)
print(f"Accuracy : {originalAcc*100:.2f}%")

Accuracy : 90.06%


In [5]:
X_selected = df[selected_features]

X_train, X_test, y_train, y_test = train_test_split(
    X_selected,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

model.fit(X_train, y_train)

pearceAcc = accuracy_score(
    y_test,
    model.predict(X_test)
)
print(f"Accuracy : {pearceAcc*100:.2f}%")

Accuracy : 90.64%


In [6]:
dt_pca = DecisionTreeClassifier(
    max_depth=MAX_DEPTH,
    random_state=RANDOM_STATE
)

dt_pca.fit(X_train_pca, y_train)

y_pred_pca = dt_pca.predict(X_test_pca)

pcaAcc = accuracy_score(y_test, y_pred_pca)
print(f"Accuracy : {pcaAcc*100:.2f}%")

Accuracy : 92.40%


In [8]:
print("Accuracy Comparison")
print(f"Before Feature Selection : {originalAcc*100:.2f}%")
print(f"Pearce Correlation Selection  : {pearceAcc*100:.2f}%")
print(f"PCA Selection  : {pcaAcc*100:.2f}%")

Accuracy Comparison
Before Feature Selection : 90.06%
Pearce Correlation Selection  : 90.64%
PCA Selection  : 92.40%
